# Exploración del catálogo Data360 — 6 visualizaciones

Prototipo de las **6 visualizaciones de exploración** de `docs/plans/01-tp-final-plan.md` §4.5:

1. Treemap jerárquico L0 → L1 → L2.
2. Top-N bases por cantidad de indicadores.
3. Heatmap base × región.
4. Dumbbell fin nominal vs. último dato real.
5. Barras apiladas de periodicidad.
6. Scatter indicadores vs. cobertura mediana.

Lee únicamente los CSV de `data/derived/`; no vuelve a consultar la API. Los gráficos son interactivos (tooltips) y se pueden exportar a HTML en una etapa posterior.

In [36]:
from pathlib import Path

import altair as alt
import pandas as pd
import squarify

pd.set_option("display.max_columns", 50)
alt.data_transformers.enable("default")


def find_project_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "data" / "derived" / "database_summary.csv").exists():
            return candidate
    raise FileNotFoundError("No se encontró data/derived/database_summary.csv")


ROOT = find_project_root()
DERIVED = ROOT / "data" / "derived"
print("Proyecto:", ROOT)
print("CSV:", sorted(p.name for p in DERIVED.glob("*.csv")))

Proyecto: /Users/selewaut/Code Projects/master/world_bank_infoviz
CSV: ['database_actual_freshness.csv', 'database_doc_counts.csv', 'database_region_coverage.csv', 'database_summary.csv', 'database_topics.csv', 'freshness_summary.csv', 'granular_freshness.csv', 'indicators.csv', 'region_overall.csv', 'region_summary.csv', 'topic_summary.csv']


## 1. Treemap jerárquico L0 → L1 → L2

**Pregunta:** ¿cómo se distribuye el catálogo entre temas y subtemas?  
**Codificación:** área = cantidad de indicadores; tres niveles anidados (L0 → tema, L1 → área, L2 → subtema); color = tema L0.  
**Fuente:** `data/derived/indicators.csv` (campos `topic_l0`, `topic_l1`, `topic_l2`).

Se usa el primer valor cuando un indicador declara más de un tema. Los indicadores sin clasificar se agrupan como **Sin clasificar** para no ocultar el 40 % del catálogo que no tiene tema L0.

In [37]:
indicators = pd.read_csv(DERIVED / "indicators.csv")


def primary_topic(series: pd.Series) -> pd.Series:
    return series.fillna("").astype(str).str.split("|").str[0].str.strip()


indicators["l0"] = primary_topic(indicators["topic_l0"]).replace("", "Sin clasificar")
indicators["l1"] = primary_topic(indicators["topic_l1"]).replace("", "Sin subclasificar")
indicators["l2"] = primary_topic(indicators["topic_l2"]).replace("", "Sin detalle")

L0_ORDER = ["Prosperity", "Sin clasificar", "People", "Infrastructure", "Planet", "Digital"]
L0_COLORS = ["#4c78a8", "#b0b0b0", "#f58518", "#54a24b", "#72b7b2", "#e45756"]

W, H = 100.0, 60.0


def inset(rect, fraction):
    return (
        rect["x"] + rect["dx"] * fraction,
        rect["y"] + rect["dy"] * fraction,
        rect["dx"] * (1 - 2 * fraction),
        rect["dy"] * (1 - 2 * fraction),
    )


def treemap_layout(sizes, x, y, w, h):
    normalized = squarify.normalize_sizes(list(sizes), w, h)
    return squarify.squarify(normalized, x, y, w, h)


leaves = []
l1_boxes = []
l0_boxes = []
l0_labels = []

l0_counts = indicators.groupby("l0").size().sort_values(ascending=False)
for (l0, _), r0 in zip(l0_counts.items(), treemap_layout(l0_counts.values, 0, 0, W, H)):
    l0_boxes.append({"x": r0["x"], "y": r0["y"], "x2": r0["x"] + r0["dx"], "y2": r0["y"] + r0["dy"]})
    l0_labels.append({"x": r0["x"] + r0["dx"] / 2, "y": r0["y"] + r0["dy"] / 2, "l0": l0})
    x0, y0, w0, h0 = inset(r0, 0.010)
    sub0 = indicators[indicators["l0"] == l0]
    l1_counts = sub0.groupby("l1").size().sort_values(ascending=False)
    for (l1, _), r1 in zip(l1_counts.items(), treemap_layout(l1_counts.values, x0, y0, w0, h0)):
        l1_boxes.append({"x": r1["x"], "y": r1["y"], "x2": r1["x"] + r1["dx"], "y2": r1["y"] + r1["dy"]})
        x1, y1, w1, h1 = inset(r1, 0.015)
        sub1 = sub0[sub0["l1"] == l1]
        l2_counts = sub1.groupby("l2").size().sort_values(ascending=False)
        for (l2, _), r2 in zip(l2_counts.items(), treemap_layout(l2_counts.values, x1, y1, w1, h1)):
            leaves.append(
                {
                    "l0": l0,
                    "l1": l1,
                    "l2": l2,
                    "indicators": int(l2_counts[l2]),
                    "x": r2["x"],
                    "y": r2["y"],
                    "x2": r2["x"] + r2["dx"],
                    "y2": r2["y"] + r2["dy"],
                    "width": r2["dx"],
                    "height": r2["dy"],
                }
            )

treemap_df = pd.DataFrame(leaves)
treemap_df["cx"] = (treemap_df["x"] + treemap_df["x2"]) / 2
treemap_df["cy"] = (treemap_df["y"] + treemap_df["y2"]) / 2
treemap_df["label"] = treemap_df.apply(
    lambda row: f"{row['l2'][:24]}\n{row['indicators']:,}" if row["width"] * row["height"] >= 12 else "",
    axis=1,
)
l1_df = pd.DataFrame(l1_boxes)
l0_df = pd.DataFrame(l0_boxes)
l0_label_df = pd.DataFrame(l0_labels)

print(f"Leaves L2: {len(treemap_df)} · indicadores: {treemap_df['indicators'].sum():,}")
print(treemap_df.groupby("l0")["indicators"].sum().sort_values(ascending=False).to_string())

treemap = (
    alt.Chart(treemap_df)
    .mark_rect(stroke="white", strokeWidth=0.5)
    .encode(
        x=alt.X("x:Q", scale=alt.Scale(domain=[0, W]), axis=None),
        x2="x2:Q",
        y=alt.Y("y:Q", scale=alt.Scale(domain=[0, H]), axis=None),
        y2="y2:Q",
        color=alt.Color(
            "l0:N",
            title="Tema L0",
            scale=alt.Scale(domain=L0_ORDER, range=L0_COLORS),
        ),
        tooltip=[
            alt.Tooltip("l0:N", title="Tema L0"),
            alt.Tooltip("l1:N", title="Tema L1"),
            alt.Tooltip("l2:N", title="Tema L2"),
            alt.Tooltip("indicators:Q", title="Indicadores", format=","),
        ],
    )
)

l1_borders = (
    alt.Chart(l1_df)
    .mark_rect(fillOpacity=0, stroke="white", strokeWidth=1)
    .encode(
        x=alt.X("x:Q", scale=alt.Scale(domain=[0, W])),
        x2="x2:Q",
        y=alt.Y("y:Q", scale=alt.Scale(domain=[0, H])),
        y2="y2:Q",
    )
)

l0_borders = (
    alt.Chart(l0_df)
    .mark_rect(fillOpacity=0, stroke="#1b1b1b", strokeWidth=2)
    .encode(
        x=alt.X("x:Q", scale=alt.Scale(domain=[0, W])),
        x2="x2:Q",
        y=alt.Y("y:Q", scale=alt.Scale(domain=[0, H])),
        y2="y2:Q",
    )
)

leaf_labels = (
    alt.Chart(treemap_df[treemap_df["label"] != ""])
    .mark_text(fontSize=8, color="white", lineBreak="\n", limit=140)
    .encode(
        x=alt.X("cx:Q", scale=alt.Scale(domain=[0, W])),
        y=alt.Y("cy:Q", scale=alt.Scale(domain=[0, H])),
        text=alt.Text("label:N"),
    )
)

l0_text = (
    alt.Chart(l0_label_df)
    .mark_text(fontSize=13, fontWeight="bold", color="#1b1b1b")
    .encode(
        x=alt.X("x:Q", scale=alt.Scale(domain=[0, W])),
        y=alt.Y("y:Q", scale=alt.Scale(domain=[0, H])),
        text=alt.Text("l0:N"),
    )
)

(
    treemap + l1_borders + l0_borders + leaf_labels + l0_text
).properties(
    width=900,
    height=540,
    title="Estructura temática del catálogo (área = indicadores; borde negro = L0, blanco = L1)",
)

Leaves L2: 190 · indicadores: 9,859
l0
Prosperity        4400
Sin clasificar    3922
People             825
Infrastructure     287
Planet             283
Digital            142


alt.LayerChart(...)

## 2. Top-N bases por cantidad de indicadores

**Pregunta:** ¿qué bases concentran el catálogo?  
**Codificación:** barras horizontales, `x = indicadores`, `y = base`, `color = tema L1 dominante`.  
**Fuente:** `data/derived/database_summary.csv`.

In [38]:
databases = pd.read_csv(DERIVED / "database_summary.csv")


def dominant_l1(value: str) -> str:
    if not isinstance(value, str) or not value.strip():
        return "Sin clasificar"
    return value.split("|")[0].split(":")[0].strip()


databases["dominant_l1"] = databases["top_l1_topics"].map(dominant_l1)
top20 = databases.sort_values("indicators", ascending=False).head(20).copy()

total = databases["indicators"].sum()
share = top20["indicators"].sum() / total
print(f"Bases: {len(databases):,} · indicadores: {total:,}")
print(f"Top 20 = {top20['indicators'].sum():,} indicadores ({share:.1%} del total)")
print(top20[["database_id", "indicators", "coverage_median", "dominant_l1"]].to_string(index=False))

bars = (
    alt.Chart(top20)
    .mark_bar(cornerRadiusEnd=3)
    .encode(
        x=alt.X("indicators:Q", title="Cantidad de indicadores"),
        y=alt.Y("database_id:N", sort="-x", title=None),
        color=alt.Color("dominant_l1:N", title="Tema L1 dominante"),
        tooltip=[
            alt.Tooltip("database_id:N", title="Base"),
            alt.Tooltip("database_name:N", title="Nombre"),
            alt.Tooltip("indicators:Q", title="Indicadores", format=","),
            alt.Tooltip("coverage_median:Q", title="Cobertura mediana"),
            alt.Tooltip("dominant_l1:N", title="Tema L1"),
        ],
    )
    .properties(width=660, height=460, title="Top 20 bases por cantidad de indicadores")
)

labels = bars.mark_text(align="left", dx=4, fontSize=10).encode(
    text=alt.Text("indicators:Q", format=","), color=alt.value("#333333")
)

(bars + labels).configure_axis(grid=True, gridOpacity=0.2)

Bases: 171 · indicadores: 9,859
Top 20 = 7,447 indicadores (75.5% del total)
database_id  indicators  coverage_median                           dominant_l1
     WB_WDI        1545            183.0                             Education
    IMF_BOP        1146             50.0                       Economic Policy
 WB_EDSTATS        1072             96.0                             Education
      WB_ES         584            180.0 Trade, Investment and Competitiveness
    IMF_IFS         495             23.0                               Finance
      WB_GS         363            217.0                                Gender
  WB_FINDEX         280             90.0                               Finance
IMF_GFS_SOO         218            151.0                       Economic Policy
     BS_SGI         193             41.0                          Institutions
     WB_HNP         192            217.0        Health, Nutrition & Population
   IMF_FSIC         184             62.0              

alt.LayerChart(...)

## 3. Heatmap base × región

**Pregunta:** ¿qué bases tienen vacíos regionales?  
**Codificación:** color = cobertura media (0–1) de los indicadores de la base en cada región.  
**Fuente:** `data/derived/database_region_coverage.csv`.

Para que sea legible se muestran las **25 bases con más indicadores** más las 8 del shortlist temático. Las 171 bases × 9 regiones completas quedan en el CSV.

El color se agrupa en **5 intervalos discretos** (sin gradiente continuo).

In [39]:
SHORTLIST = {
    "PEERING_DB": "Datacenters",
    "PCH_IXP": "IXPs",
    "OOKLA_ST": "Velocidad",
    "ITU_DH": "Conectividad",
    "WB_WDI": "Internet / energía",
    "WB_SE4ALL": "Energía",
    "WB_RISE": "Energía (política)",
    "WB_PPI": "Inversión infra",
}

REGION_ORDER = ["AFE", "AFW", "EAS", "ECS", "LCN", "MEA", "NAC", "SAS", "SSF"]

region_cov = pd.read_csv(DERIVED / "database_region_coverage.csv")
heatmap_ids = list(
    dict.fromkeys(
        databases.sort_values("indicators", ascending=False).head(25)["database_id"].tolist()
        + list(SHORTLIST)
    )
)
heatmap_df = region_cov[region_cov["database_id"].isin(heatmap_ids)].copy()
heatmap_df["shortlist"] = heatmap_df["database_id"].isin(SHORTLIST)

COVERAGE_LABELS = ["0–20 %", "20–40 %", "40–60 %", "60–80 %", "80–100 %"]
COVERAGE_COLORS = ["#FFFFCC", "#A1DAB4", "#41B6C4", "#2C7FB8", "#253494"]
heatmap_df["coverage_band"] = pd.cut(
    heatmap_df["mean_coverage"],
    bins=[-0.001, 0.2, 0.4, 0.6, 0.8, 1.0],
    labels=COVERAGE_LABELS,
)

print(f"Bases en el heatmap: {len(heatmap_ids)} · celdas: {len(heatmap_df)}")
print(heatmap_df.groupby("database_id")["mean_coverage"].mean().sort_values().head(8).to_string())

heat = (
    alt.Chart(heatmap_df)
    .mark_rect(stroke="white", strokeWidth=0.5)
    .encode(
        x=alt.X("region:N", sort=REGION_ORDER, title=None),
        y=alt.Y("database_id:N", sort=heatmap_ids, title=None),
        color=alt.Color(
            "coverage_band:N",
            title="Cobertura media",
            sort=COVERAGE_LABELS,
            scale=alt.Scale(
                domain=COVERAGE_LABELS,
                range=COVERAGE_COLORS,
                type="ordinal",
            ),
        ),
        tooltip=[
            alt.Tooltip("database_id:N", title="Base"),
            alt.Tooltip("region_name:N", title="Región"),
            alt.Tooltip("coverage_band:N", title="Intervalo"),
            alt.Tooltip("mean_coverage:Q", title="Cobertura media", format=".3f"),
            alt.Tooltip("median_coverage:Q", title="Cobertura mediana", format=".3f"),
            alt.Tooltip("indicators:Q", title="Indicadores"),
            alt.Tooltip("region_countries:Q", title="Países en la región"),
        ],
    )
)

heat_text = heat.mark_text(fontSize=7).encode(
    text=alt.Text("mean_coverage:Q", format=".0%"),
    color=alt.condition(
        alt.datum.mean_coverage > 0.6, alt.value("white"), alt.value("#333333")
    ),
)

(heat + heat_text).properties(
    width=620,
    height=620,
    title="Cobertura media por base × región (top 25 + shortlist)",
)

Bases en el heatmap: 32 · celdas: 288
database_id
BS_SGI        0.159778
IMF_IRFCL     0.203778
IMF_QGFS      0.257333
IMF_IFS       0.303222
IMF_BOP       0.343222
IMF_GFS_BS    0.354222
GI_AII        0.362333
IMF_FSIC      0.364111


alt.LayerChart(...)

## 4. Dumbbell: fin nominal vs. último dato real

**Pregunta:** ¿la metadata promete datos más recientes de los que realmente existen?  
**Codificación:** un par de puntos por base (fin nominal y último dato observado), unidos por una línea; se muestran las 30 bases con mayor brecha absoluta.  
**Fuente:** `data/derived/database_actual_freshness.csv`.

- `gap = nominal_end − actual_latest_year`.
- `gap > 0` → la metadata declara un fin posterior al último dato real (desactualización / proyección).
- `gap < 0` → hay datos más allá del fin declarado (metadata conservadora).

In [40]:
freshness = pd.read_csv(DERIVED / "database_actual_freshness.csv")
freshness["actual_year"] = freshness["actual_latest_year"].astype(str).str.extract(r"(\d{4})").astype(int)
freshness["nominal_year"] = freshness["nominal_end"].astype(str).str.extract(r"(\d{4})").astype(int)
freshness["gap"] = freshness["nominal_year"] - freshness["actual_year"]
freshness["estado"] = freshness["gap"].map(
    lambda g: "Metadata sobreestima" if g > 0 else ("Metadata subestima" if g < 0 else "Coincide")
)

print("Bases sondeadas:", len(freshness))
print(freshness["estado"].value_counts().to_string())
print("\nMayores brechas:")
print(
    freshness.reindex(freshness["gap"].abs().sort_values(ascending=False).index)
    .head(15)[["database_id", "actual_latest_period", "nominal_end", "gap", "estado"]]
    .to_string(index=False)
)

shown = freshness.reindex(freshness["gap"].abs().sort_values(ascending=False).index).head(30)
order = shown.sort_values("gap")["database_id"].tolist()

long = shown.melt(
    id_vars=["database_id", "gap", "estado"],
    value_vars=["actual_year", "nominal_year"],
    var_name="serie",
    value_name="year",
)
long["serie"] = long["serie"].map(
    {"actual_year": "Último dato real", "nominal_year": "Fin nominal (metadata)"}
)

rule = (
    alt.Chart(shown)
    .mark_rule(color="#b0b0b0", strokeWidth=2)
    .encode(
        y=alt.Y("database_id:N", sort=order, title=None),
        x=alt.X("actual_year:Q", title="Año", scale=alt.Scale(zero=False)),
        x2="nominal_year:Q",
    )
)

points = (
    alt.Chart(long)
    .mark_circle(size=90, opacity=0.9)
    .encode(
        y=alt.Y("database_id:N", sort=order, title=None),
        x=alt.X("year:Q", title="Año", scale=alt.Scale(zero=False)),
        color=alt.Color(
            "serie:N",
            title=None,
            scale=alt.Scale(
                domain=["Último dato real", "Fin nominal (metadata)"],
                range=["#1f77b4", "#d62728"],
            ),
        ),
        tooltip=[
            alt.Tooltip("database_id:N", title="Base"),
            alt.Tooltip("serie:N", title="Serie"),
            alt.Tooltip("year:Q", title="Año"),
            alt.Tooltip("gap:Q", title="Brecha (años)"),
            alt.Tooltip("estado:N", title="Estado"),
        ],
    )
)

(rule + points).properties(
    width=620, height=640, title="Fin nominal vs. último dato real — 30 mayores brechas"
)

Bases sondeadas: 158
estado
Coincide                114
Metadata sobreestima     39
Metadata subestima        5

Mayores brechas:
 database_id actual_latest_period nominal_end  gap               estado
IMF_GFS_SSUC                 2014        2025   11 Metadata sobreestima
IMF_GFS_SFCP                 2014        2025   11 Metadata sobreestima
  IMF_GFS_BS                 2014        2025   11 Metadata sobreestima
      WB_PIP                 2014        2025   11 Metadata sobreestima
    WB_BOOST                 2020        2030   10 Metadata sobreestima
    IMF_QGFS              2016-Q2     2026-Q2   10 Metadata sobreestima
     WB_LPGD                 2013        2023   10 Metadata sobreestima
   IMF_FSIRE                 2014        2023    9 Metadata sobreestima
  UNESCO_UIS                 2018        2026    8 Metadata sobreestima
      EPC_AI                 2018        2025    7 Metadata sobreestima
      WB_SHP                 2017        2022    5 Metadata sobreestima
     I

alt.LayerChart(...)

## 5. Barras apiladas de periodicidad

**Pregunta:** ¿qué bases del catálogo no son puramente anuales?  
**Codificación:** composición porcentual de cada base; se muestran **solo las 13 bases que no son 100 % anuales** (de 171). Las 158 restantes se omiten porque una barra completamente anual no aporta información.  
**Fuente:** `data/derived/indicators.csv` (`periodicity`) + `database_summary.csv` (nombre legible).

Cada barra se etiqueta con el **nombre legible** de la base (el código queda en el tooltip). Las periodicidades se muestran sin agrupar y traducidas al español.

**Ad hoc (irregular)** = series sin frecuencia fija, publicadas según disponibilidad. En este catálogo son solo 4 indicadores de WDI provenientes de Enterprise Surveys.

El color es **único por tipo de periodicidad** y se repite en todas las barras: la paleta recorre la dimensión de periodicidad, no la base.

In [41]:
PERIODICITY_COLORS = {
    "Anual": "#5B7C99",
    "Desconocida": "#B7B7B7",
    "Trienal": "#D99A6C",
    "Quinquenal": "#7FA88B",
    "Mensual": "#C48B9F",
    "Trimestral": "#93B7C9",
    "Bienal": "#C97A6D",
    "Cada dos años": "#D9C58F",
    "Ad hoc (irregular)": "#9C8573",
    "Otra": "#9B8BB4",
}

READABLE_NAMES = {
    "WB_BPS": "COVID-19 Business Pulse Surveys",
    "WRI_AQDT": "Aqueduct 4.0 Water Risk",
    "IPC_IPC": "IPC Food Security Classification",
    "UNCTAD_LSC": "Liner Shipping Connectivity Index",
    "WB_FINDEX": "Global Findex Database",
    "WB_LPI": "Logistics Performance Index",
    "IMF_GHW": "IMF Global Housing Watch",
    "FAO_CP": "FAO Consumer Price Indices",
    "WB_CSC": "WBG Corporate Scorecard",
    "WB_ID4D": "ID4D Identification Dataset",
    "OECD_AHD": "OECD Affordable Housing Database",
    "WB_WDI": "World Development Indicators",
    "WB_HCP": "Human Capital Project",
}

PERIODICITY_LABELS = {
    "Annual": "Anual",
    "Triennial": "Trienal",
    "Biennial": "Bienal",
    "Every two years": "Cada dos años",
    "Five-Year Interval": "Quinquenal",
    "Ad hoc [adhoc]": "Ad hoc (irregular)",
    "Monthly": "Mensual",
    "Quarterly": "Trimestral",
    "Other": "Otra",
    "Unknown": "Desconocida",
}

per_base = (
    indicators.groupby(["database_id", "periodicity"]).size().reset_index(name="count")
)
base_totals = per_base.groupby("database_id")["count"].sum()
shares = (
    per_base.pivot_table(
        index="database_id", columns="periodicity", values="count", fill_value=0
    )
    .div(base_totals, axis=0)
    .fillna(0)
)
shares["annual_share"] = shares.get("Annual", 0)
shares["quarterly_share"] = shares.get("Quarterly", 0)
shares["monthly_share"] = shares.get("Monthly", 0)
shares["other_share"] = 1 - shares[
    ["annual_share", "quarterly_share", "monthly_share"]
].sum(axis=1)
non_annual = shares[shares["annual_share"] < 1].copy()
non_annual["has_quarterly"] = (non_annual["quarterly_share"] > 0).astype(int)
non_annual["has_monthly"] = (non_annual["monthly_share"] > 0).astype(int)
non_annual = non_annual.sort_values(
    ["annual_share", "has_quarterly", "has_monthly", "other_share", "database_id"],
    ascending=[False, False, False, False, True],
)
annual_share = shares["annual_share"]
non_annual_ids = non_annual.index.tolist()

names = databases.set_index("database_id")["database_name"].to_dict()


def db_label(database_id: str) -> str:
    if database_id in READABLE_NAMES:
        return READABLE_NAMES[database_id]
    name = str(names.get(database_id, "")).strip()
    return name if name and name != database_id else database_id


per_non = per_base[per_base["database_id"].isin(non_annual_ids)].copy()
per_non["database_label"] = per_non["database_id"].map(db_label)
per_non["periodicity_label"] = per_non["periodicity"].map(
    lambda value: PERIODICITY_LABELS.get(value, value)
)
per_non["database_name"] = per_non["database_id"].map(db_label)

label_order = [db_label(database_id) for database_id in non_annual_ids]
PERIODICITY_DOMAIN = [
    "Anual",
    "Trimestral",
    "Mensual",
    "Trienal",
    "Bienal",
    "Cada dos años",
    "Quinquenal",
    "Ad hoc (irregular)",
    "Otra",
    "Desconocida",
]
present_labels = set(per_non["periodicity_label"])
periodicity_order = [label for label in PERIODICITY_DOMAIN if label in present_labels]
periodicity_palette = [PERIODICITY_COLORS[label] for label in periodicity_order]

print(f"Bases con periodicidad no 100 % anual: {len(non_annual_ids)} de {len(base_totals)}")
print(
    non_annual[["annual_share", "quarterly_share", "monthly_share", "other_share"]]
    .assign(base=[db_label(database_id) for database_id in non_annual.index])
    .rename(
        columns={
            "annual_share": "anual",
            "quarterly_share": "trimestral",
            "monthly_share": "mensual",
            "other_share": "otras",
        }
    )
    .to_string()
)

chart = (
    alt.Chart(per_non)
    .mark_bar()
    .encode(
        y=alt.Y(
            "database_label:N",
            sort=label_order,
            title=None,
            axis=alt.Axis(labelLimit=260),
        ),
        x=alt.X("count:Q", stack="normalize", title="Proporción de indicadores"),
        color=alt.Color(
            "periodicity_label:N",
            title="Periodicidad",
            scale=alt.Scale(domain=periodicity_order, range=periodicity_palette),
        ),
        tooltip=[
            alt.Tooltip("database_id:N", title="Código"),
            alt.Tooltip("database_name:N", title="Nombre"),
            alt.Tooltip("periodicity_label:N", title="Periodicidad"),
            alt.Tooltip("count:Q", title="Indicadores", format=","),
        ],
    )
    .properties(
        width=760,
        height=len(non_annual_ids) * 32 + 60,
        title=alt.TitleParams(
            text=f"Bases con periodicidad no 100 % anual ({len(non_annual_ids)} de {len(base_totals)})",
            subtitle=f"Se omiten las {len(base_totals) - len(non_annual_ids)} bases cuyos indicadores son todos anuales",
        ),
    )
)

chart

Bases con periodicidad no 100 % anual: 13 de 171
periodicity     anual  trimestral  mensual     otras                               base
database_id                                                                            
WB_HCP       0.976744    0.000000      0.0  0.023256              Human Capital Project
WB_WDI       0.947573    0.000000      0.0  0.052427       World Development Indicators
OECD_AHD     0.944444    0.055556      0.0  0.000000   OECD Affordable Housing Database
WB_ID4D      0.363636    0.000000      0.0  0.636364        ID4D Identification Dataset
WB_CSC       0.142857    0.000000      0.0  0.857143            WBG Corporate Scorecard
IMF_GHW      0.000000    1.000000      0.0  0.000000           IMF Global Housing Watch
UNCTAD_LSC   0.000000    1.000000      0.0  0.000000  Liner Shipping Connectivity Index
FAO_CP       0.000000    0.000000      1.0  0.000000         FAO Consumer Price Indices
IPC_IPC      0.000000    0.000000      1.0  0.000000   IPC Food Securit

alt.Chart(...)

## 6. Scatter amplitud vs. profundidad

**Pregunta:** ¿las bases con más indicadores también cubren más economías?  
**Codificación:** `x = indicadores` (escala log), `y = cobertura mediana`, `tamaño = cobertura máxima`, color = pertenencia al shortlist temático.  
**Fuente:** `data/derived/database_summary.csv`.

El shortlist son las 8 bases del recorte datacenter / conectividad / energía / infraestructura (`PEERING_DB`, `PCH_IXP`, `OOKLA_ST`, `ITU_DH`, `WB_WDI`, `WB_SE4ALL`, `WB_RISE`, `WB_PPI`).

In [42]:
databases["grupo"] = databases["database_id"].map(
    lambda db: "Shortlist temático" if db in SHORTLIST else "Resto del catálogo"
)
databases["shortlist_role"] = databases["database_id"].map(SHORTLIST).fillna("")

print(databases[databases["database_id"].isin(SHORTLIST)][
    ["database_id", "shortlist_role", "indicators", "coverage_median", "coverage_max"]
].sort_values("indicators", ascending=False).to_string(index=False))

base = (
    alt.Chart(databases)
    .mark_circle(opacity=0.65, stroke="white", strokeWidth=0.5)
    .encode(
        x=alt.X(
            "indicators:Q",
            title="Cantidad de indicadores (escala log)",
            scale=alt.Scale(type="log"),
        ),
        y=alt.Y("coverage_median:Q", title="Cobertura mediana (economías)"),
        size=alt.Size(
            "coverage_max:Q",
            title="Cobertura máxima",
            scale=alt.Scale(range=[20, 450]),
        ),
        color=alt.Color(
            "grupo:N",
            title=None,
            scale=alt.Scale(
                domain=["Shortlist temático", "Resto del catálogo"],
                range=["#d62728", "#9aa5b1"],
            ),
        ),
        tooltip=[
            alt.Tooltip("database_id:N", title="Base"),
            alt.Tooltip("shortlist_role:N", title="Rol en el shortlist"),
            alt.Tooltip("indicators:Q", title="Indicadores", format=","),
            alt.Tooltip("coverage_median:Q", title="Cobertura mediana"),
            alt.Tooltip("coverage_max:Q", title="Cobertura máxima"),
        ],
    )
)

labels = (
    alt.Chart(databases[databases["database_id"].isin(SHORTLIST)])
    .mark_text(dx=7, dy=-5, fontSize=10, fontWeight="bold")
    .encode(
        x="indicators:Q",
        y="coverage_median:Q",
        text="database_id:N",
        color=alt.value("#a4161a"),
    )
)

(base + labels).properties(
    width=660, height=440, title="Amplitud vs. profundidad del catálogo (shortlist resaltado)"
).interactive()

database_id     shortlist_role  indicators  coverage_median  coverage_max
     WB_WDI Internet / energía        1545            183.0           217
     ITU_DH       Conectividad          39            189.0           230
    WB_RISE Energía (política)          27            140.0           140
  WB_SE4ALL            Energía           8            219.5           228
     WB_PPI    Inversión infra           5            112.0           133
   OOKLA_ST          Velocidad           2            242.0           244
    PCH_IXP               IXPs           2            156.0           156
 PEERING_DB        Datacenters           1            158.0           158


alt.LayerChart(...)

## Notas para la versión final

- Las seis figuras cubren el catálogo desde ángulos complementarios; las más narrativas siguen siendo **#4 (frescura)** y **#6 (selección del shortlist)**.
- Hallazgos: top 20 bases = 75,5 % de los indicadores; 44/158 bases con fin nominal ≠ dato real (39 sobreestiman, 5 subestiman); 95,3 % de los indicadores son anuales.
- Antes de exportar: fijar tipografía/formato, títulos en lenguaje natural y nota de fuente al pie.
- El treemap incluye ~40 % de indicadores sin tema L0: es un hallazgo de clasificación, no un error del gráfico.